# LIGHTGBM


## Crescimento leaf-wise e level-wise
No crescimento level-wise, a árvore expande nós na mesma profundidade. LightGBM, por padrão, cresce leaf-wise: a cada passo divide a folha com maior ganho. Para um mesmo número de folhas, isso pode reduzir a perda mais rapidamente, mas pode aumentar overfitting; `max_depth`, `num_leaves` e `min_child_samples` ajudam a controlar a complexidade.

## Histogramas
Valores contínuos são agrupados em bins discretos e os ganhos de divisão são avaliados nesses histogramas, reduzindo custo de memória e computação. A discretização é uma aproximação do espaço de cortes.

## Diferenças para XGBoost
Ambos implementam boosting de árvores e suportam histogramas, regularização e interfaces scikit-learn. LightGBM destaca crescimento leaf-wise e categorias nativas em colunas categóricas pandas; XGBoost usa seus próprios parâmetros e configuração de categóricas. Os nomes semelhantes não implicam semântica idêntica: por exemplo, LightGBM `subsample` requer `subsample_freq` maior que zero para ativar bagging.

### Hiperparametros e controle de overfitting
- `num_leaves`: máximo de folhas por árvore; valores maiores aumentam capacidade. Em árvores com `max_depth` positivo, considerar `num_leaves <= 2^max_depth`; a recomendação prática pode ser ainda menor para limitar complexidade leaf-wise.
- `max_depth`: limita profundidade; valor não positivo não impõe limite. Usar em conjunto com `num_leaves`.
- `learning_rate`: encolhe contribuição de cada árvore; taxas menores geralmente exigem mais árvores.
- `n_estimators`: número de iterações/árvores; valores altos elevam custo e podem sobreajustar sem controle.
- `min_child_samples`: número mínimo aproximado de observações por folha; aumentar restringe divisões e reduz variância.
- `subsample`: fração de observações para bagging. Em LightGBM, é necessário `subsample_freq > 0`; abaixo ativamos frequência 1.
- `colsample_bytree`: fração de atributos usada por árvore; valores menores adicionam aleatoriedade e podem reduzir overfitting.
- `reg_alpha` e `reg_lambda`: penalidades L1 e L2 sobre pesos; valores maiores regularizam mais.

A busca combina esses controles, com limites de folhas coerentes com a profundidade amostrada. `min_child_weight` também restringe soma mínima de Hessianos por folha.




In [5]:
%pip install numpy pandas lightgbm optuna scikit-learn pyarrow

Note: you may need to restart the kernel to use updated packages.


In [6]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (average_precision_score, confusion_matrix,
                             precision_score, recall_score, f1_score)


In [7]:
import os
pathKaggle = "../input/datasets/luisfilipelima/bancotransacao/"
path = "../../data/processed/"
pathDrive = "/content/drive/MyDrive/TreinosIAS/XGBoosting/dados/"
#path=pathDrive
path = pathKaggle
path


'../input/datasets/luisfilipelima/bancotransacao/'

In [8]:
import pyarrow as pa
import pyarrow.parquet as pq

print("Pandas:", pd.__version__)
print("PyArrow:", pa.__version__)

obt = pd.read_parquet(path + "obt_with_terminal_history.parquet")
obtTrain = pd.read_parquet(path + "obt_train.parquet")
obtTest = pd.read_parquet(path + "obt_validation.parquet")

obtTrain.head()


Pandas: 2.3.3
PyArrow: 23.0.1


,transaction_id,tx_datetime,tx_amount,card_id,terminal_id,terminal_category,hour,day_week,terminal_age_days,is_fraud,...,terminal_tx_count_24h,terminal_distinct_cards_24h,seconds_since_last_terminal_tx,card_history_count_30d,card_amount_mean_30d,card_amount_std_30d,amount_z_score_30d,card_distinct_terminals,terminal_fraud_rate,card_fraud_rate
0,1156634,2018-02-15 13:34:00,12.400000,3520842735991929,847,SmartTalk Communications,13,4,173.0,False,...,24,23,1396.0,165,12.378606,8.682511,0.002464,155,0.001805,0.003953
1,1157370,2018-02-15 13:57:03,52.340000,370934368869147,847,SmartTalk Communications,13,4,173.0,False,...,25,24,1383.0,123,105.470894,91.224616,-0.582418,134,0.001803,0.000000
2,1157375,2018-02-15 13:57:17,78.169998,6011689217447071,847,SmartTalk Communications,13,4,173.0,False,...,26,25,14.0,122,139.657131,132.426997,-0.464310,168,0.001802,0.000000
3,1159536,2018-02-15 15:09:34,23.850000,2662393784087193,847,SmartTalk Communications,15,4,173.0,False,...,24,23,4337.0,142,91.744367,119.426299,-0.568504,149,0.001800,0.000000
4,1160251,2018-02-15 15:35:49,29.459999,3515828436568584,847,SmartTalk Communications,15,4,173.0,False,...,24,23,1575.0,102,43.886078,27.739705,-0.520052,113,0.001799,0.016575


In [9]:
cat_features = ["terminal_category"]

features = [
    "tx_amount", "hour", "day_week", "terminal_age_days",
    "terminal_fraud_rate", "card_fraud_rate", "card_distinct_terminals",
    "terminal_new_for_card", "card_tx_count_1h", "card_tx_count_24h",
    "card_distinct_terminals_24h", "seconds_since_last_card_tx",
    "terminal_tx_count_1h", "terminal_tx_count_24h",
    "terminal_distinct_cards_24h", "seconds_since_last_terminal_tx",
    "card_history_count_30d", "card_amount_mean_30d",
    "card_amount_std_30d", "amount_z_score_30d", "terminal_category",
]
target = "is_fraud"

def prepararParaLightGBM(df):
    df["datetime"] = pd.to_datetime(df["tx_datetime"], errors="raise")
    df[target] = df[target].astype(int)
    df["terminal_category"] = (
        df["terminal_category"].astype("object").fillna("SEM_CATEGORIA").astype("category")
    )
    return df.sort_values("tx_datetime").reset_index(drop=True)

obt = prepararParaLightGBM(obt)
obtTrain = prepararParaLightGBM(obtTrain)
obtTest = prepararParaLightGBM(obtTest)


In [10]:
def calcular_lucro(TransacoesAceitas):
    if TransacoesAceitas.empty:
        print("Nenhuma transação aceita; lucro = 0.")
        return 0.0
    legitimas = TransacoesAceitas["is_fraud"].eq(0)
    fraudulentas = TransacoesAceitas["is_fraud"].eq(1)
    receita = 0.03 * TransacoesAceitas.loc[legitimas, "tx_amount"].sum()
    prejuizo = TransacoesAceitas.loc[fraudulentas, "tx_amount"].sum()
    print("CALCULANDO MELHOR LUCRO...")
    print(f"VALOR:{float(receita - prejuizo)}")
    return float(receita - prejuizo).__round__(2)

def classificarTransacoes(x, y, prob, limiar):
    transacoes = x.copy()
    transacoes["is_fraud"] = np.asarray(y, dtype=int)
    transacoes["probabilidade_fraude"] = prob
    aceita = transacoes["probabilidade_fraude"] < limiar
    return transacoes.loc[aceita].copy(), transacoes.loc[~aceita].copy()

def encontrar_melhor_limiar(probabilidades, X_val, y_val):
    valores = X_val["tx_amount"].to_numpy()
    y = np.asarray(y_val, dtype=int)
    ordem = np.argsort(probabilidades, kind="stable")
    p_ord = probabilidades[ordem]
    lucro = np.where(y[ordem] == 0, 0.03 * valores[ordem], -valores[ordem])
    lucro_acumulado = np.cumsum(lucro)
    fim_grupo = np.concatenate([p_ord[1:] != p_ord[:-1], [True]])
    indices = np.flatnonzero(fim_grupo)
    lucros = np.concatenate([[0.0], lucro_acumulado[indices]])
    pos = int(np.argmax(lucros))
    limiar = (np.nextafter(p_ord[0], -np.inf) if pos == 0
              else np.nextafter(p_ord[indices[pos - 1]], np.inf))
    return float(limiar), float(lucros[pos])


In [ ]:
%pip install torch

In [12]:
import torch

print("CUDA disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Quantidade de GPUs:", torch.cuda.device_count())
    print("GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo selecionado:", device)

CUDA disponível: True
Quantidade de GPUs: 2
GPU: Tesla T4
Dispositivo selecionado: cuda


In [19]:
SEED = 42
N_INIT_TRIALS = 10
N_TRIALS = 30
N_SPLITS = 10

Buscatrain = obtTrain.tail(300_000).copy()
X_Train = Buscatrain[features].copy().reset_index(drop=True)
y_Train = Buscatrain[target].copy().reset_index(drop=True)
tscv = TimeSeriesSplit(n_splits=N_SPLITS)

fixed_params = {
    "device_type":"gpu",
    "objective": "binary",
    "metric": "average_precision",
    "random_state": SEED,
    "max_bin": 63,
    "n_jobs": 4,
    "verbosity": -1,
}

def preparar_fold_lgbm(X_fit, X_val):
    # Alinha categorias sem usar rótulos e preserva ordem temporal.
    X_fit, X_val = X_fit.copy(), X_val.copy()
    for coluna in cat_features:
        categorias = pd.Index(pd.concat([X_fit[coluna], X_val[coluna]], ignore_index=True).unique())
        X_fit[coluna] = pd.Categorical(X_fit[coluna], categories=categorias)
        X_val[coluna] = pd.Categorical(X_val[coluna], categories=categorias)
    return X_fit, X_val

def objetivo(trial):
    max_depth = trial.suggest_int("max_depth", 4, 8)
    params = {
        "n_estimators": 1000,
        "max_depth": max_depth,
        "num_leaves": trial.suggest_int("num_leaves", 8, min(2 ** max_depth, 128)),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.20, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "min_child_weight": trial.suggest_float("min_child_weight", 1e-3, 10.0, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "subsample_freq": 1,
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 100.0, log=True),
    }
    lucros_folds, prauc_folds = [], []
    for train_idx, val_idx in tscv.split(X_Train):
        X_fit, X_val = X_Train.iloc[train_idx], X_Train.iloc[val_idx]
        y_fit, y_val = y_Train.iloc[train_idx], y_Train.iloc[val_idx]
        X_fit, X_val = preparar_fold_lgbm(X_fit, X_val)
        model = lgb.LGBMClassifier(**fixed_params, **params)
        model.fit(X_fit, y_fit, categorical_feature=cat_features)
        probabilidades = model.predict_proba(X_val)[:, 1]
        _, lucro_fold = encontrar_melhor_limiar(probabilidades, X_val, y_val)
        lucros_folds.append(lucro_fold)
        prauc_folds.append(average_precision_score(y_val, probabilidades))
    trial.set_user_attr("lucros_por_fold", lucros_folds)
    trial.set_user_attr("prauc_media", float(np.mean(prauc_folds)))
    return float(np.mean(lucros_folds))

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED, n_startup_trials=N_INIT_TRIALS))
study.optimize(objetivo, n_trials=N_TRIALS, n_jobs=1, show_progress_bar=True)


[I 2026-11-11 00:02:45,736] A new study created in memory with name: no-name-22f92ac6-dd5d-45a3-b556-bddce080d059


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-11-11 00:04:17,544] Trial 0 finished with value: 50393.579296875 and parameters: {'max_depth': 5, 'num_leaves': 31, 'learning_rate': 0.08960785365368121, 'min_child_samples': 64, 'min_child_weight': 0.004207988669606638, 'subsample': 0.662397808134481, 'colsample_bytree': 0.6232334448672797, 'reg_alpha': 0.6245760287469893, 'reg_lambda': 1.0129197956845732}. Best is trial 0 with value: 50393.579296875.
[I 2026-11-11 00:05:22,673] Trial 1 finished with value: 50338.85078125 and parameters: {'max_depth': 7, 'num_leaves': 10, 'learning_rate': 0.18276027831785724, 'min_child_samples': 85, 'min_child_weight': 0.0070689749506246055, 'subsample': 0.6727299868828402, 'colsample_bytree': 0.6733618039413735, 'reg_alpha': 5.472429642032198e-06, 'reg_lambda': 0.4205156450913872}. Best is trial 0 with value: 50393.579296875.
[I 2026-11-11 00:06:51,026] Trial 2 finished with value: 50394.46484375 and parameters: {'max_depth': 6, 'num_leaves': 24, 'learning_rate': 0.06252287916406217, 'min_ch

In [20]:
print("Lucro médio dos folds da melhor configuração:", study.best_value)
print("Melhores hiperparâmetros:", study.best_params)
print("PRAUC médio nos folds:", study.best_trial.user_attrs["prauc_media"])
print("Lucros por fold:", study.best_trial.user_attrs["lucros_por_fold"])


Lucro médio dos folds da melhor configuração: 50493.591796875
Melhores hiperparâmetros: {'max_depth': 5, 'num_leaves': 18, 'learning_rate': 0.02631930091744587, 'min_child_samples': 53, 'min_child_weight': 0.0057562847708749615, 'subsample': 0.6321457314116755, 'colsample_bytree': 0.6228479046156681, 'reg_alpha': 3.1990525187010875e-05, 'reg_lambda': 7.8200891221204625}
PRAUC médio nos folds: 0.5107438429264415
Lucros por fold: [51068.23828125, 51653.33984375, 49607.00390625, 48897.09375, 49627.234375, 48296.12890625, 52071.01953125, 49523.9921875, 52868.1484375, 51323.71875]


In [21]:
# Obtém previsões walk-forward da melhor configuração para selecionar o limiar em treino.
melhores_Parametros = study.best_params.copy()
probabilidades_oof, y_oof, X_oof = [], [], []
for train_idx, val_idx in tscv.split(X_Train):
    X_fit, X_val = X_Train.iloc[train_idx], X_Train.iloc[val_idx]
    y_fit, y_val = y_Train.iloc[train_idx], y_Train.iloc[val_idx]
    X_fit, X_val = preparar_fold_lgbm(X_fit, X_val)
    modelo_fold = lgb.LGBMClassifier(**fixed_params, **melhores_Parametros)
    modelo_fold.fit(X_fit, y_fit, categorical_feature=cat_features)
    probabilidades_oof.extend(modelo_fold.predict_proba(X_val)[:, 1])
    y_oof.extend(y_val.to_numpy())
    X_oof.append(X_val)
X_oof = pd.concat(X_oof, ignore_index=True)
y_oof = np.asarray(y_oof, dtype=int)
probabilidades_oof = np.asarray(probabilidades_oof)
limiar_final, lucro_oof = encontrar_melhor_limiar(probabilidades_oof, X_oof, y_oof)
print("Limiar selecionado em validação walk-forward de obtTrain:", limiar_final)
print("Lucro OOF:", lucro_oof)

x_test, y_test = obtTest[features].copy(), obtTest[target].copy()
modelo_final = lgb.LGBMClassifier(**fixed_params, **melhores_Parametros)
modelo_final.fit(Buscatrain[features], Buscatrain[target], categorical_feature=cat_features)


Limiar selecionado em validação walk-forward de obtTrain: 0.1527757369614505
Lucro OOF: 503343.9375


LGBMClassifier(colsample_bytree=0.6228479046156681, device_type='gpu',
               learning_rate=0.02631930091744587, max_bin=63, max_depth=5,
               metric='average_precision', min_child_samples=53,
               min_child_weight=0.0057562847708749615, n_jobs=4, num_leaves=18,
               objective='binary', random_state=42,
               reg_alpha=3.1990525187010875e-05, reg_lambda=7.8200891221204625,
               subsample=0.6321457314116755, verbosity=-1)

In [22]:
# Avaliação final única em obtTest, com limiar definido em obtTrain.
probabilidades_teste = modelo_final.predict_proba(x_test)[:, 1]
predicoes_teste = (probabilidades_teste >= limiar_final).astype(int)
resultado_teste = x_test.copy()
resultado_teste["probabilidade_fraude"] = probabilidades_teste
resultado_teste["fraude_prevista"] = predicoes_teste
resultado_teste["fraude_real"] = y_test.to_numpy()
TransacoesAceitas, TransacoesRecusadas = classificarTransacoes(x_test, y_test, probabilidades_teste, limiar_final)
lucro_teste = calcular_lucro(TransacoesAceitas)
matriz = confusion_matrix(y_test, predicoes_teste)
precisao = precision_score(y_test, predicoes_teste, zero_division=0)
recall = recall_score(y_test, predicoes_teste, zero_division=0)
prauc_teste = average_precision_score(y_test, probabilidades_teste)
print("Matriz de confusão:\n", matriz)
print("Lucro em maio:", lucro_teste)
print("Taxa de transações recusadas:", len(TransacoesRecusadas) / len(x_test))
print("Precisão:", precisao, "Recall:", recall, "PR-AUC:", prauc_teste)


CALCULANDO MELHOR LUCRO...
VALOR:1774888.125
Matriz de confusão:
 [[1041818       0]
 [   8104    5130]]
Lucro em maio: 1774888.12
Taxa de transações recusadas: 0.0048623195823523395
Precisão: 1.0 Recall: 0.3876379023726764 PR-AUC: 0.46624636566863664


In [23]:
for parametro, valor in melhores_Parametros.items():
    print(f"{parametro}: {valor}")


max_depth: 5
num_leaves: 18
learning_rate: 0.02631930091744587
min_child_samples: 53
min_child_weight: 0.0057562847708749615
subsample: 0.6321457314116755
colsample_bytree: 0.6228479046156681
reg_alpha: 3.1990525187010875e-05
reg_lambda: 7.8200891221204625
